In [ ]:
import cupy as cp
import numpy as np
import os
import sys
from core import *
import time
from joint_posterior import JointPosterior, JointZoomConfig
from joint_estimator import JointEstimator
from joint_kl import choose_next_bias

In [44]:
DATA_DIR = r"C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Simulation\Dataset_10"
#By scan 
scan_index = 3
bz_prior_range = 2
sigma_noise  = 0.4
#start time, curr_time
curr_time = 0.0
DATASET3 = DataContext(
                            sim_time = os.path.join(DATA_DIR, "t_array.csv"),
                            sim_freq = os.path.join(DATA_DIR, "delz_MHz.csv"),
                        sim_by = os.path.join(DATA_DIR, "dely_MHz.csv"),
                        sim_intensities = sorted([
                            os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.startswith("transH_t_vs_Bz_alpha2500_delx_0_dely_") and f.endswith(".csv")
                        ], key=lambda x: float(x.split("_dely_")[1].split(".csv")[0])),
                        exp_time = r"C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\t_array_ Copy.csv",
                        exp_freq_axis = r"C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\Bz_Y_MHz.csv",
                        exp_data = f"C:\\Users\\Dinesh\\UGP\\bayesian_nmor\\DataFiles_to_Dinesh_Pranav\\Data_files\\Experiment\\26_03_2026_CW_data\\Bz_V_Readout_By_{scan_index}.csv",
                        interpolator=r"C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Simulation\Dataset_10\gpu_sim_interpolator_new_normalisation_linear_spline_0.3_threshold.npz",
                        Sim_Aligned = True,
                        Exp_Aligned = False,
                        sim_pulse_thresh=0.3, 
                        exp_pulse_thresh=0.3
                        )
DEFAULT_ALTOPT_PARAMS = ParameterContext(
                            num_iter=10,
                            curr_time = curr_time,
                            max_time = 70.0,
                            tol_bz=1e-20,
                            tol_by=1e-20,
                            t_step = 0.2,   
                            fixed_by_estimate=0.0,
                            fixed_bz_estimate=0.0,
                            B_unk_bound_transverse_upper=0.5,
                            B_unk_bound_longitudinal = bz_prior_range,
                            print_plot = False,
                            likelihood_mode_longitudinal="Gaussian",
                            likelihood_mode_transverse="Gaussian",
                            sigma_noise_longitudinal = sigma_noise,
                            sigma_noise_transverse = sigma_noise
                            )

In [45]:
t_sim, f_sim, by_sim, full_interp = get_final_interpolator(DATASET3, DEFAULT_ALTOPT_PARAMS)
t_exp, f_bias_axis, exp_matrix = load_experiment(DATASET3, DATASET3.Exp_Aligned)
t_exp, f_bias_axis, exp_matrix = cp.asarray(t_exp), cp.asarray(f_bias_axis), cp.asarray(exp_matrix)


# bz_support = cp.arange(-DEFAULT_ALTOPT_PARAMS.B_unk_bound_longitudinal, DEFAULT_ALTOPT_PARAMS.B_unk_bound_longitudinal, DEFAULT_ALTOPT_PARAMS.init_resolution_longitudinal)  # type: ignore
# curr_res_z = DEFAULT_ALTOPT_PARAMS.init_resolution_longitudinal
# by_support = cp.arange(DEFAULT_ALTOPT_PARAMS.B_unk_bound_transverse_lower, DEFAULT_ALTOPT_PARAMS.B_unk_bound_transverse_upper, DEFAULT_ALTOPT_PARAMS.init_resolution_transverse)  # type: ignore
# curr_res_y = DEFAULT_ALTOPT_PARAMS.init_resolution_transverse
bz_support = cp.linspace(-DEFAULT_ALTOPT_PARAMS.B_unk_bound_longitudinal, DEFAULT_ALTOPT_PARAMS.B_unk_bound_longitudinal, 100)  # setting to 100 for simplicity

by_support = cp.linspace(DEFAULT_ALTOPT_PARAMS.B_unk_bound_transverse_lower, DEFAULT_ALTOPT_PARAMS.B_unk_bound_transverse_upper, 100)  # setting to 100 for simplicity


curr_bias = 0.0
start_time = DEFAULT_ALTOPT_PARAMS.curr_time

...Building Final Interpolator...
Time taken to load Interpolator: 2.9402966499328613
Loading Experiment from C:\Users\Dinesh\UGP\bayesian_nmor\DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\Bz_V_Readout_By_3.csv...
  > Exp Start: -0.1000s
0.23102164268493652 was the time taken to load exp data.


In [46]:
ZOOM_CONFIG = JointZoomConfig(
    credible_mass=0.999,
    trigger_span_ratio=0.25,
    margin_cells=3,
    growth_factor=1.5,
    max_grid_points_per_axis=None,  # Set an explicit cap to permit grid growth.
)

posterior = JointPosterior(
    bz_support,
    by_support,
)

estimator = JointEstimator(
    posterior,
    full_interp,
    DEFAULT_ALTOPT_PARAMS,
    zoom_config=ZOOM_CONFIG,
)

Joint Bayesian Update Loop - We'll turn this into a nice function later once this executes.

In [47]:

history = {
    "time": [],
    "map_bz": [],
    "map_by": [],
    "std_bz": [],
    "std_by": [],
    "bias": [],
    "expectedkl": [],
    "posteriors": [],
    "grid_shapes": [],
    "bz_axes": [],
    "by_axes": [],
    "zoom_events": [],
    "covariance_matrix": [],
}

time_cursor = DEFAULT_ALTOPT_PARAMS.curr_time
curr_bias = 0.0

start_wall = time.time()
loop_index = 0

while time_cursor < DEFAULT_ALTOPT_PARAMS.max_time:

    # ---------------------------------------------------------
    # Current control interval
    # ---------------------------------------------------------
    print(loop_index)
    loop_index += 1

    t_next = time_cursor + DEFAULT_ALTOPT_PARAMS.t_step

    idx_start = cp.searchsorted(
        t_exp,
        cp.asarray(time_cursor),
    )

    idx_end = cp.searchsorted(
        t_exp,
        cp.asarray(t_next),
    )

    if idx_start >= idx_end:
        break

    # ---------------------------------------------------------
    # Snap bias to nearest available experiment column
    # ---------------------------------------------------------

    bias_idx = cp.abs(
        f_bias_axis - curr_bias
    ).argmin()

    curr_bias = f_bias_axis[bias_idx]

    # ---------------------------------------------------------
    # Experimental measurement
    # ---------------------------------------------------------

    y_obs = exp_matrix[
        idx_start:idx_end,
        bias_idx,
    ]

    t_chunk = t_exp[
        idx_start:idx_end
    ]

    # ---------------------------------------------------------
    # Bayesian update
    # ---------------------------------------------------------

    summary = estimator.update(
        measurement=y_obs,
        times=t_chunk,
        bias=curr_bias,
    )

    # ---------------------------------------------------------
    # Save history
    # ---------------------------------------------------------

    history["time"].append(float(t_next))

    history["map_bz"].append(
        float(summary.map_bz)
    )

    history["map_by"].append(
        float(summary.map_by)
    )

    history["std_bz"].append(
        float(summary.std_bz)
    )

    history["std_by"].append(
        float(summary.std_by)
    )

    history["covariance_matrix"].append(
        summary.covariance.get()
    )

    history["bias"].append(
        float(curr_bias)
    )

    history["posteriors"].append(
        estimator.posterior.weights.get()
    )

    support_state = estimator.support_history[-1]
    history["grid_shapes"].append(support_state["grid_shape"])
    history["bz_axes"].append(support_state["bz_axis"])
    history["by_axes"].append(support_state["by_axis"])

    #FIXME are we matching the altest posterior with the latest support?
    
    if (
        estimator.zoom_events
        and estimator.zoom_events[-1]["update_index"] == len(estimator.update_records)
    ):
        history["zoom_events"].append(estimator.zoom_events[-1])

    print(
        f"T={t_next:.2f} "
        f"| Bias={curr_bias:.3f} "
        f"| MAP=({summary.map_bz:.4f}, "
        f"{summary.map_by:.4f}) "
        f"| Std=({summary.std_bz:.4f}, "
        f"{summary.std_by:.4f})"
    )

    # ---------------------------------------------------------
    # Candidate bias range
    # ---------------------------------------------------------

    f_index_1 = cp.where(
        f_bias_axis > f_sim[10]
    )[0][0]

    f_index_2 = cp.where(
        f_bias_axis > f_sim[-10]
    )[0][0]

    candidate_biases = f_bias_axis[
        f_index_1:f_index_2
    ]

    # ---------------------------------------------------------
    # Adaptive design
    # ---------------------------------------------------------

    curr_bias, expectedkl = choose_next_bias(
        posterior=estimator.posterior,
        candidate_biases=candidate_biases,
        current_time=t_next,
        next_time=t_next + DEFAULT_ALTOPT_PARAMS.t_step,
        interpolator=full_interp,
        params=DEFAULT_ALTOPT_PARAMS,
    )

    history["expectedkl"].append(
        expectedkl.get()
    )

    # ---------------------------------------------------------
    # Advance replay
    # ---------------------------------------------------------

    time_cursor = t_next

print(
    f"\nFinished in "
    f"{time.time()-start_wall:.2f} s"
)

0
T=0.20 | Bias=0.000 | MAP=(-0.3434, 0.0606) | Std=(1.1103, 0.1475)
1
T=0.40 | Bias=1.740 | MAP=(0.6667, 0.0657) | Std=(1.0585, 0.1443)
2
T=0.60 | Bias=-1.874 | MAP=(0.1010, 0.3384) | Std=(1.0811, 0.1402)
3
T=0.80 | Bias=-1.250 | MAP=(-0.0202, 0.1616) | Std=(1.0449, 0.1390)
4
T=1.00 | Bias=1.250 | MAP=(0.0606, 0.1616) | Std=(1.0053, 0.1377)
5
T=1.20 | Bias=-1.651 | MAP=(-0.0606, 0.1616) | Std=(0.9920, 0.1360)
6
T=1.40 | Bias=-1.250 | MAP=(-0.0606, 0.1616) | Std=(0.9318, 0.1356)
7
T=1.60 | Bias=1.740 | MAP=(0.0606, 0.1616) | Std=(0.8989, 0.1323)
8
T=1.80 | Bias=-0.134 | MAP=(0.0606, 0.1212) | Std=(1.0063, 0.1333)
9
T=2.00 | Bias=0.045 | MAP=(0.0202, 0.1616) | Std=(1.1329, 0.1340)
10
T=2.20 | Bias=-0.045 | MAP=(0.0202, 0.1212) | Std=(1.1486, 0.1322)
11
T=2.40 | Bias=-0.089 | MAP=(0.0606, 0.1212) | Std=(0.9662, 0.1296)
12
T=2.60 | Bias=-0.179 | MAP=(0.0606, 0.1212) | Std=(1.0808, 0.1232)
13
T=2.80 | Bias=0.045 | MAP=(0.0202, 0.1616) | Std=(1.2141, 0.1107)
14
T=3.00 | Bias=-0.045 | MAP=(0

Lets save the history.

In [48]:
np.savez_compressed(
        f"by_{scan_index}_estimation.npz",
        **history
    )

In [49]:
print(history["zoom_events"])

[{'update_index': 44, 'old_shape': (100, 100), 'new_shape': (100, 100), 'old_bz_bounds': (-2.0, 2.0), 'old_by_bounds': (0.0, 0.5), 'new_bz_bounds': (0.018898664059954554, 0.0619094167481266), 'new_by_bounds': (0.17448680351906162, 0.24975562072336266), 'hpd_bounds': (0.020202020202020374, 0.06060606060606078, 0.1767676767676768, 0.2474747474747475), 'zoomed_bz': True, 'zoomed_by': True}, {'update_index': 294, 'old_shape': (100, 100), 'new_shape': (100, 100), 'old_bz_bounds': (0.018898664059954554, 0.0619094167481266), 'old_by_bounds': (0.17448680351906162, 0.24975562072336266), 'new_bz_bounds': (0.032030327555822004, 0.04312987663664059), 'new_by_bounds': (0.17448680351906162, 0.24975562072336266), 'hpd_bounds': (0.03236667752796802, 0.042793526664494576, 0.2178233952427501, 0.2368306723145433), 'zoomed_bz': True, 'zoomed_by': False}, {'update_index': 295, 'old_shape': (100, 100), 'new_shape': (100, 100), 'old_bz_bounds': (0.032030327555822004, 0.04312987663664059), 'old_by_bounds': (0

In [66]:
len(history["posteriors"])

350

In [67]:
history

NpzFile 'by_3_estimation.npz' with keys: time, map_bz, map_by, std_bz, std_by...

In [69]:
animation_posterior(posteriors = history["posteriors"], bz_grid=history["bz_axes"], by_grid=history["by_axes"], save_path = f"posterior_animation_by_{scan_index}.gif")

In [54]:
print(r'MAP $B_z$ :',history["map_bz"][-1]*100/0.7, r'$\\$ MAP $B_y$ :',history["map_by"][-1]*100/0.7, r'$\\$ Variance Covariance Matrix : $\\$', history["covariance_matrix"][-1]*(100/0.7)*100/0.7)

MAP $B_z$ : 5.392611011757758 $\\$ MAP $B_y$ : 32.687261095169525 $\\$ Variance Covariance Matrix : $\\$ [[ 0.03676801 -0.00130479]
 [-0.00130479  0.11354374]]


1. By_0: $\\$
MAP $B_z$ :  7.01029421112670 $\\$
MAP $B_y$ : 9.425314370981306 $\\$
Variance Covariance Matrix : $\\$ 
[[0.0098283  0.00084535] $\\$
[0.00084535 0.01532049]]

2. By_1: $\\$
MAP $B_z$ : 8.471814923427852 $\\$ MAP $B_y$ : 21.98415677318702 $\\$ Variance Covariance Matrix : $\\$ [[8.42484085e-04 9.85868086e-06] $\\$
 [9.85868086e-06 2.46032838e-03]]

3. By_2: $\\$
MAP $B_z$ : 4.111277784534833 $\\$ MAP $B_y$ : 26.9482896954281 $\\$ Variance Covariance Matrix : $\\$ [[ 0.02441509 -0.00308086] $\\$
 [-0.00308086  0.06217547]]

4. By_3: $\\$
MAP $B_z$ : 5.392611011757758 $\\$ MAP $B_y$ : 32.687261095169525 $\\$ Variance Covariance Matrix : $\\$ [[ 0.03676801 -0.00130479] $\\$
 [-0.00130479  0.11354374]]
